# Transliteration by using LSTM Encoder-Decoder

In [2]:
# import neccessary libraries
import os
import pandas as pd
import zipfile
import json
import requests
# from google.colab import drive

In [3]:
# Mount Google Drive if you want to save the dataset there
# drive.mount('/content/drive')

In [4]:
# Make sure the URL points directly to the zip file content.
hin_zip_url = "https://huggingface.co/datasets/ai4bharat/Aksharantar/resolve/main/hin.zip"

# Define the local path to save the zip file
zip_file_path = "hin.zip"

In [5]:
print(f"Downloading {zip_file_path}...")
response = requests.get(hin_zip_url)
if response.status_code == 200:
    with open(zip_file_path, 'wb') as f:
        f.write(response.content)
    print(f"Downloaded {zip_file_path} successfully.")
else:
    print(f"Failed to download {zip_file_path}. Status code: {response.status_code}")

Downloaded hin.zip successfully.


In [6]:
print(f"Unzipping {zip_file_path}...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall('.') # Extract to the current directory
print("Unzipping completed.")

Unzipping hin.zip...
Unzipping completed.


In [7]:
# List the extracted files to confirm structure
extracted_files = os.listdir('.')
hin_files = [f for f in extracted_files if f.startswith('hin') and f.endswith('.json')]
print(f"Extracted Hindi files: {hin_files}")

Extracted Hindi files: ['hin_test.json', 'hin_train.json', 'hin_valid.json']


In [8]:
# Function to load a single JSON file into a pandas DataFrame
def load_json_to_df(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # Each line is a valid JSON object
            data.append(json.loads(line))
    return pd.DataFrame(data)

In [43]:
# Load the training, validation, and test datasets
# Assuming the files are named hin_train.json, hin_test.json, hin_valid.json
# Adjust the names based on the actual extracted files
train_file = 'hin_train.json'
valid_file = 'hin_valid.json'
test_file = 'hin_test.json'

# Check if files exist before loading
train_df = load_json_to_df(train_file) if os.path.exists(train_file) else pd.DataFrame()
valid_df = load_json_to_df(valid_file) if os.path.exists(valid_file) else pd.DataFrame()
test_df = load_json_to_df(test_file) if os.path.exists(test_file) else pd.DataFrame()

train_df = train_df[:100000]
valid_df = valid_df[:5000]
test_df = test_df[:5000]

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {valid_df.shape}")
print(f"Test set shape: {test_df.shape}")

Training set shape: (100000, 5)
Validation set shape: (5000, 5)
Test set shape: (5000, 4)


In [44]:
# Display the first few rows of the training set to confirm structure
if not train_df.empty:
    print("\nFirst few rows of the training set:")
    print(train_df.head())
    print("\nColumns in training set:", train_df.columns.tolist())
else:
    print(f"\nWarning: {train_file} was not found or is empty.")


First few rows of the training set:
  unique_identifier native word english word    source  score
0              hin1    जन्मदिवस   janamdivas  Dakshina    NaN
1              hin2       रक्खा        rakha  Dakshina    NaN
2              hin3    मिलीजुली     milijuli  Dakshina    NaN
3              hin4      जांचों     jaanchon  Dakshina    NaN
4              hin5       चमकता     chamkata  Dakshina    NaN

Columns in training set: ['unique_identifier', 'native word', 'english word', 'source', 'score']


In [45]:
# Display the first few rows of the validation set
if not valid_df.empty:
    print("\nFirst few rows of the validation set:")
    print(valid_df.head())
else:
    print(f"\nWarning: {valid_file} was not found or is empty.")


First few rows of the validation set:
  unique_identifier native word english word    source  score
0              hin1      स्पाइक        spike  Wikidata    NaN
1              hin2     त्रिलोक       trilok  Wikidata    NaN
2              hin3        चंदा       chanda  Wikidata    NaN
3              hin4        मीता        meeta  Wikidata    NaN
4              hin5         जैक         jack  Wikidata    NaN


In [46]:
# Display the first few rows of the test set
if not test_df.empty:
    print("\nFirst few rows of the test set:")
    print(test_df.head())
else:
    print(f"\nWarning: {test_file} was not found or is empty.")


First few rows of the test set:
  unique_identifier      native word    english word   source
0              hin1    मैट्रोलॉजिस्ट    maitrologist  AK-Freq
1              hin2  पीएचडब्ल्यूसीएस           phwcs  AK-Freq
2              hin3  प्रतिद्वन्दियों  pratidwandiyon  AK-Freq
3              hin4      प्रतियुक्ति      pratiyukti  AK-Freq
4              hin5       एक्सिसटेंस      eksisatens  AK-Freq


In [47]:
SRC_COLUMN = 'native word'  # Hindi Devanagari script
TGT_COLUMN = 'english word' # English Roman script

# ------------------------------
# 4. Build Character Vocabularies
# ------------------------------
def build_vocab(series, name):
    """
    Builds vocabulary from a pandas Series of strings.

    Args:
        series (pd.Series): Series containing strings.
        name (str): Name for the vocabulary (e.g., 'Source', 'Target').

    Returns:
        tuple: A tuple containing:
            - char_to_idx (dict): Mapping from character to index.
            - idx_to_char (dict): Mapping from index to character.
            - vocab (list): The sorted vocabulary list including special tokens.
    """
    chars = set()
    for word in series:
        # Ensure word is a string before processing
        if isinstance(word, str):
            chars.update(list(word))
        else:
            print(f"Warning: Non-string value found in {name} series: {word}. Skipping.")
    vocab = sorted(list(chars))
    # Add special tokens at the beginning
    special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
    vocab = special_tokens + vocab
    char_to_idx = {ch: i for i, ch in enumerate(vocab)}
    idx_to_char = {i: ch for i, ch in enumerate(vocab)}
    print(f"{name} vocab size: {len(vocab)}")
    return char_to_idx, idx_to_char, vocab

In [48]:
# Source: Hindi Devanagari script (input)
print("Building source vocabulary...")
src_char_to_idx, src_idx_to_char, src_vocab = build_vocab(train_df[SRC_COLUMN], "Source (Hindi Devanagari)")

# Target: English Roman script (output)
print("Building target vocabulary...")
tgt_char_to_idx, tgt_idx_to_char, tgt_vocab = build_vocab(train_df[TGT_COLUMN], "Target (English Roman)")

Building source vocabulary...
Source (Hindi Devanagari) vocab size: 72
Building target vocabulary...
Target (English Roman) vocab size: 30


In [49]:
# Optional: Save vocabularies if needed later
import pickle
with open('src_char_to_idx.pkl', 'wb') as f:
    pickle.dump(src_char_to_idx, f)
with open('tgt_char_to_idx.pkl', 'wb') as f:
    pickle.dump(tgt_char_to_idx, f)
with open('tgt_idx_to_char.pkl', 'wb') as f:
    pickle.dump(tgt_idx_to_char, f)

In [50]:
# Optional: Print a few examples of the mappings
print("\nExample Source (Hindi) mappings:")
print(f"  'श' -> {src_char_to_idx.get('श', '<NOT_FOUND>')}")
print(f"  '<SOS>' -> {src_char_to_idx.get('<SOS>', '<NOT_FOUND>')}")
print(f"  Index 0 -> '{src_idx_to_char.get(0, '<NOT_FOUND>')}'")

print("\nExample Target (English) mappings:")
print(f"  's' -> {tgt_char_to_idx.get('s', '<NOT_FOUND>')}")
print(f"  '<EOS>' -> {tgt_char_to_idx.get('<EOS>', '<NOT_FOUND>')}")
print(f"  Index 1 -> '{tgt_idx_to_char.get(1, '<NOT_FOUND>')}'")


Example Source (Hindi) mappings:
  'श' -> 50
  '<SOS>' -> 1
  Index 0 -> '<PAD>'

Example Target (English) mappings:
  's' -> 22
  '<EOS>' -> 2
  Index 1 -> '<SOS>'


In [51]:
import torch
import torch.nn as nn
import json
import os

# Ensure the data directory exists for saving config
os.makedirs("data", exist_ok=True)

# Special token indices (must match the vocabulary building step)
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

In [52]:
# ------------------------------
# 2. Hyperparameters
# ------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_DIM = 128
HID_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 128 # Note: This is typically used in the training loop, not model definition

# Use the vocabularies built previously
SRC_VOCAB_SIZE = len(src_vocab)
TGT_VOCAB_SIZE = len(tgt_vocab)

print(f"Device: {DEVICE}")
print(f"Source vocab size: {SRC_VOCAB_SIZE}, Target vocab size: {TGT_VOCAB_SIZE}")

Device: cpu
Source vocab size: 72, Target vocab size: 30


In [53]:
# ------------------------------
# 3. Model Definition
# ------------------------------

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.emb_dim = emb_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout if n_layers > 1 else 0, batch_first=True) # Common practice: no dropout if only 1 layer
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.dropout(self.embedding(src))  # [batch_size, src_len, emb_dim]
        outputs, (hidden, cell) = self.lstm(embedded)  # outputs: [batch_size, src_len, hid_dim]
        # outputs are usually ignored in basic seq2seq, context is passed via hidden/cell
        return hidden, cell  # hidden, cell: [n_layers, batch_size, hid_dim]


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.emb_dim = emb_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout if n_layers > 1 else 0, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        """
        input: LongTensor of token ids. Allowed shapes:
               - [batch]           (recommended)
               - [batch, 1]        (common from some callers)
               - any shape with a trailing singleton dims -> will be flattened to [batch]
        hidden, cell: LSTM states with shapes [num_layers * num_directions, batch, hid_dim]
        """
        # Ensure input is 1-D: [batch]
        if input.dim() > 1:
            # collapse any extra dims into a single batch dimension
            # e.g. [batch,1] -> [batch], or [batch,1,1] -> [batch]
            input = input.view(-1)

        # ensure dtype and device are correct
        if not input.dtype == torch.long:
            input = input.long()
        input = input.to(next(self.parameters()).device)

        # embed and add the time dimension -> [batch, 1, emb_dim]
        embedded = self.embedding(input).unsqueeze(1)

        # LSTM expects 3D input: [batch, seq_len, emb_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        # output: [batch, 1, hid_dim] -> squeeze time -> [batch, hid_dim]
        output = output.squeeze(1)

        # map to vocab: [batch, vocab_size]
        prediction = self.fc_out(output) # Corrected from self.out

        return prediction, hidden, cell

    # def forward(self, input, hidden, cell):
    #     # input: [batch_size] (current token)
    #     # hidden, cell: [n_layers, batch_size, hid_dim]
    #     input = input.unsqueeze(1)  # [batch_size, 1]
    #     embedded = self.dropout(self.embedding(input))  # [batch_size, 1, emb_dim]
    #     output, (hidden, cell) = self.lstm(embedded, (hidden, cell))  # output: [batch_size, 1, hid_dim]
    #     prediction = self.fc_out(output.squeeze(1))  # [batch_size, output_dim]
    #     return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, "Hidden dimensions must match!"
        assert encoder.n_layers == decoder.n_layers, "Number of layers must match!"

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len]
        # trg: [batch_size, trg_len] (includes SOS and EOS)
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # Encode source
        hidden, cell = self.encoder(src)

        # First input to decoder is SOS token
        input = torch.full((batch_size,), SOS_IDX, dtype=torch.long, device=self.device)

        for t in range(1, trg_len): # Start from 1 because 0th token is SOS
            # Pass through decoder
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            # Decide whether to use teacher forcing
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)  # [batch_size]

            # Next input is either ground truth or predicted token
            input = trg[:, t] if teacher_force else top1

        return outputs  # [batch_size, trg_len, output_dim]

In [54]:
# ------------------------------
# 4. Instantiate Model
# ------------------------------

enc = Encoder(SRC_VOCAB_SIZE, EMB_DIM, HID_DIM, NUM_LAYERS, DROPOUT)
dec = Decoder(TGT_VOCAB_SIZE, EMB_DIM, HID_DIM, NUM_LAYERS, DROPOUT)

model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 1,863,966 trainable parameters


In [55]:
# Save model config for inference later
model_config = {
    "emb_dim": EMB_DIM,
    "hid_dim": HID_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "src_vocab_size": SRC_VOCAB_SIZE,
    "tgt_vocab_size": TGT_VOCAB_SIZE,
    "pad_idx": PAD_IDX,
    "sos_idx": SOS_IDX,
    "eos_idx": EOS_IDX,
    "unk_idx": UNK_IDX
}

config_path = "data/model_config.json"
with open(config_path, "w") as f:
    json.dump(model_config, f)
print(f"Model configuration saved to {config_path}")

# Print model architecture
print("\nModel Architecture:")
print(model)

Model configuration saved to data/model_config.json

Model Architecture:
Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(72, 128, padding_idx=0)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(30, 128, padding_idx=0)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
    (fc_out): Linear(in_features=256, out_features=30, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
)


In [56]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import time
import os

In [57]:
# ------------------------------
# 2. Custom Dataset Class
# ------------------------------

class TransliterationDataset(Dataset):
    def __init__(self, df, src_char_to_idx, tgt_char_to_idx, unk_idx=UNK_IDX):
        self.df = df
        self.src_char_to_idx = src_char_to_idx
        self.tgt_char_to_idx = tgt_char_to_idx
        self.unk_idx = unk_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src_word = row['native word'] # Replace with your actual column name if different
        tgt_word = row['english word'] # Replace with your actual column name if different

        # Tokenize source (Hindi Devanagari)
        src_indices = [self.src_char_to_idx.get(ch, self.unk_idx) for ch in src_word]

        # Tokenize target (English Roman) with SOS and EOS
        tgt_indices = [SOS_IDX] + [self.tgt_char_to_idx.get(ch, self.unk_idx) for ch in tgt_word] + [EOS_IDX]

        return torch.tensor(src_indices, dtype=torch.long), torch.tensor(tgt_indices, dtype=torch.long)

In [58]:
# ------------------------------
# 3. Collate Function (for padding)
# ------------------------------

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    # Pad sequences
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)

    return src_padded, tgt_padded

In [59]:
# ------------------------------
# 4. Create DataLoaders
# ------------------------------

# Assuming you have train_df, valid_df loaded
# If you want to subsample the training data as per assignment instructions (e.g., to 100k examples):
# max_train_examples = 100000
# if len(train_df) > max_train_examples:
#     train_df = train_df.sample(n=max_train_examples, random_state=42).reset_index(drop=True)
#     print(f"Subsampled training data to {len(train_df)} examples.")

train_dataset = TransliterationDataset(train_df, src_char_to_idx, tgt_char_to_idx)
val_dataset = TransliterationDataset(valid_df, src_char_to_idx, tgt_char_to_idx) # Assuming valid_df exists

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 782, Val batches: 40


In [60]:
# ------------------------------
# 5. Define Training Components
# ------------------------------

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [61]:
# ------------------------------
# 6. Training & Evaluation Functions
# ------------------------------

def train_epoch(model, dataloader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0
    for src, trg in dataloader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)

        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio=0.5)  # [batch, trg_len, vocab]

        # Reshape for loss calculation: (N*T, V) vs (N*T,)
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)  # Skip the first time step (SOS)
        trg = trg[:, 1:].reshape(-1)  # Skip the first time step (SOS) in target as well

        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


def evaluate(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in dataloader:
            src, trg = src.to(DEVICE), trg.to(DEVICE)
            # Turn off teacher forcing during evaluation
            output = model(src, trg, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim) # Skip the first time step (SOS)
            trg = trg[:, 1:].reshape(-1) # Skip the first time step (SOS) in target

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [ ]:
# ------------------------------
# 7. Training Loop
# ------------------------------

N_EPOCHS = 10
best_valid_loss = float('inf')
model_save_dir = "models"
os.makedirs(model_save_dir, exist_ok=True)
best_model_path = os.path.join(model_save_dir, "best_lstm_model.pt")

print("Starting training...\n")
for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    valid_loss = evaluate(model, val_loader, criterion)

    end_time = time.time()
    epoch_mins = int((end_time - start_time) / 60)
    epoch_secs = int((end_time - start_time) % 60)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"✅ Saved best model at epoch {epoch+1}")

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.4f} | Val Loss: {valid_loss:.4f}')
    print("-" * 40)

print("\nTraining finished.")
print(f"Best validation loss: {best_valid_loss:.4f}")
print(f"Best model saved at: {best_model_path}")

Starting training...

✅ Saved best model at epoch 1
Epoch: 01 | Time: 4m 47s
	Train Loss: 1.5907 | Val Loss: 1.0059
----------------------------------------
✅ Saved best model at epoch 2
Epoch: 02 | Time: 4m 40s
	Train Loss: 0.9038 | Val Loss: 0.8774
----------------------------------------
✅ Saved best model at epoch 3
Epoch: 03 | Time: 4m 14s
	Train Loss: 0.7963 | Val Loss: 0.8603
----------------------------------------


In [29]:
import torch
import torch.nn.functional as F
import pandas as pd
import heapq # For the beam search priority queue

# Assuming model, src_char_to_idx, tgt_char_to_idx, idx_to_char, SOS_IDX, EOS_IDX, PAD_IDX, DEVICE are defined
# And the model is loaded in evaluation mode
model.eval()

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(65, 128, padding_idx=0)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(30, 128, padding_idx=0)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
    (fc_out): Linear(in_features=256, out_features=30, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
)

In [30]:
# ------------------------------
# 1. Greedy Search Inference
# ------------------------------
def greedy_search(model, src_indices, max_len=50):
    """
    Performs greedy search decoding for a single source sequence.

    Args:
        model (nn.Module): The trained Seq2Seq model.
        src_indices (torch.Tensor): Source token indices [src_len].
        max_len (int): Maximum length for the generated target sequence.

    Returns:
        list: List of token indices representing the generated target sequence (excluding SOS, up to EOS).
    """
    model.eval()
    with torch.no_grad():
        # Add batch dimension [1, src_len]
        src_tensor = src_indices.unsqueeze(0).to(DEVICE)

        # Encode the source sequence
        hidden, cell = model.encoder(src_tensor)

        # Start with SOS token
        input_token = torch.full((1,), SOS_IDX, dtype=torch.long, device=DEVICE)
        outputs = []

        for _ in range(max_len):
            output, hidden, cell = model.decoder(input_token, hidden, cell)
            # Get the token with the highest probability
            predicted_token_idx = output.argmax(1).item()
            outputs.append(predicted_token_idx)

            # Stop if EOS token is generated
            if predicted_token_idx == EOS_IDX:
                break

            # Use the predicted token as input for the next step
            input_token = torch.full((1,), predicted_token_idx, dtype=torch.long, device=DEVICE)

    # Return the sequence up to (and including) the first EOS token (or max_len)
    # Exclude the SOS token from the output sequence
    eos_idx = next((i for i, x in enumerate(outputs) if x == EOS_IDX), len(outputs))
    return outputs[:eos_idx]

In [31]:
# ------------------------------
# 2. Beam Search Inference
# ------------------------------
def beam_search(model, src_indices, beam_width=3, max_len=50):
    """
    Performs beam search decoding for a single source sequence.

    Args:
        model (nn.Module): The trained Seq2Seq model.
        src_indices (torch.Tensor): Source token indices [src_len].
        beam_width (int): Number of candidates to keep at each step.
        max_len (int): Maximum length for the generated target sequence.

    Returns:
        list: List of token indices representing the best generated target sequence (excluding SOS, up to EOS).
    """
    model.eval()
    with torch.no_grad():
        # Add batch dimension [1, src_len]
        src_tensor = src_indices.unsqueeze(0).to(DEVICE)

        # Encode the source sequence
        hidden, cell = model.encoder(src_tensor)

        # Initialize beam: (log_prob, [token_ids], hidden, cell)
        # Start with SOS token
        initial_input = torch.full((1,), SOS_IDX, dtype=torch.long, device=DEVICE)
        output, hidden, cell = model.decoder(initial_input, hidden, cell)
        log_probs = torch.log_softmax(output, dim=1) # [1, tgt_vocab_size]

        # Get top-k initial candidates
        top_k_log_probs, top_k_indices = torch.topk(log_probs.squeeze(0), beam_width)

        # Initialize the beam with top-k initial predictions
        beam = []
        for log_prob, token_idx in zip(top_k_log_probs, top_k_indices):
             # (cumulative_log_prob, [generated_token_ids], hidden, cell)
             # token_ids includes SOS initially, will be stripped later
            heapq.heappush(beam, (-log_prob.item(), [SOS_IDX, token_idx.item()], hidden, cell))

        finished_sequences = []

        for step in range(1, max_len): # Start from 1 as SOS is already added
            candidates = []

            # Expand each sequence in the current beam
            for _ in range(len(beam)):
                log_prob, seq, hidden_beam, cell_beam = heapq.heappop(beam)

                # If the last token is EOS, add the sequence to finished list
                if seq[-1] == EOS_IDX:
                    finished_sequences.append((log_prob, seq))
                    continue # Move to the next sequence in the beam

                # Get the last token to feed into the decoder
                last_token = torch.full((1,), seq[-1], dtype=torch.long, device=DEVICE)

                # Run decoder step
                output, hidden_next, cell_next = model.decoder(last_token, hidden_beam, cell_beam)
                log_probs_next = torch.log_softmax(output, dim=1).squeeze(0) # [tgt_vocab_size]

                # Get top-k expansions for this sequence
                top_k_log_probs_next, top_k_indices_next = torch.topk(log_probs_next, beam_width)

                for log_prob_add, token_idx_add in zip(top_k_log_probs_next, top_k_indices_next):
                    new_log_prob = log_prob - log_prob_add.item() # Subtract log prob (as we store -log_prob)
                    new_seq = seq + [token_idx_add.item()]
                    candidates.append((new_log_prob, new_seq, hidden_next, cell_next))

            # Add all candidates back to the heap
            for c in candidates:
                heapq.heappush(beam, c)

            # Keep only the top 'beam_width' candidates in the beam for the next step
            # If a sequence finished, it was removed from the beam, so we might have less than beam_width active sequences
            # Only keep the top beam_width *active* sequences (those not ending in EOS yet)
            # Sort beam by cumulative log probability (ascending because we stored -log_prob)
            # This is implicitly handled by the heapq structure.
            # We need to manage the finished sequences and active sequences separately for efficiency,
            # but for simplicity, we just keep the top beam_width overall at the end of each step.
            # A more efficient implementation would separate finished/unfinished.
            # For now, we just ensure the heap size doesn't exceed beam_width * max_len potentially.
            # A better approach: after expanding, get the top beam_width from the combined list.
            if len(beam) > beam_width:
                 # Get the top beam_width items (lowest -log_prob, i.e., highest prob)
                 top_beam = heapq.nsmallest(beam_width, beam)
                 beam = top_beam
                 heapq.heapify(beam) # Re-heapify the truncated list

            # Stop if we have enough finished sequences and the best unfinished is worse than the worst finished
            # This is an optimization, not strictly necessary if beam_width is small and max_len is reasonable.
            if len(finished_sequences) >= beam_width:
                 # Sort finished sequences by log probability (ascending)
                 finished_sequences.sort(key=lambda x: x[0])
                 # If the best unfinished (top of heap) has worse prob than the worst finished, stop
                 if beam and finished_sequences[-1][0] < beam[0][0]:
                     break

        # Add any remaining sequences in the beam to finished if they haven't hit max_len
        while beam:
            finished_sequences.append(heapq.heappop(beam))

        # Sort finished sequences by cumulative log probability (ascending, so best first)
        finished_sequences.sort(key=lambda x: x[0])

        # Return the best sequence (excluding the initial SOS token)
        if finished_sequences:
            best_seq = finished_sequences[0][1] # Get the token IDs
            # Find the first EOS token in the best sequence
            eos_idx = next((i for i, x in enumerate(best_seq) if x == EOS_IDX), len(best_seq))
            return best_seq[1:eos_idx] # Exclude SOS, include up to first EOS
        else:
            # Fallback if no sequence finishes (should be rare with max_len)
            # Return the best unfinished sequence up to max_len, excluding SOS
            best_seq = beam[0][1] if beam else [SOS_IDX]
            return best_seq[1:] # Exclude SOS token

In [32]:
# Example usage (assuming you have a function to convert string to indices)
def string_to_indices(text, char_to_idx, unk_idx=UNK_IDX):
    """Helper function to convert a string to a tensor of indices."""
    indices = [char_to_idx.get(ch, unk_idx) for ch in text]
    return torch.tensor(indices, dtype=torch.long)

def indices_to_string(indices, idx_to_char):
    """Helper function to convert a list of indices to a string."""
    tokens = [idx_to_char.get(idx, '<UNK>') for idx in indices]
    # Remove special tokens like <SOS>, <EOS>, <PAD> if present in output
    filtered_tokens = [t for t in tokens if t not in ['<SOS>', '<EOS>', '<PAD>']]
    return "".join(filtered_tokens)

In [33]:
# Example:
input_text = "नमस्ते"
src_tensor = string_to_indices(input_text, src_char_to_idx)

greedy_output_indices = greedy_search(model, src_tensor)
greedy_output_str = indices_to_string(greedy_output_indices, tgt_idx_to_char)
print(f"Input: {input_text}")
print(f"Greedy Output: {greedy_output_str}")

beam_output_indices = beam_search(model, src_tensor, beam_width=5)
beam_output_str = indices_to_string(beam_output_indices, tgt_idx_to_char)
print(f"Beam Search Output (beam=5): {beam_output_str}")

print("Inference functions (greedy_search, beam_search) defined.")

Input: नमस्ते
Greedy Output: namasta
Beam Search Output (beam=5): namast
Inference functions (greedy_search, beam_search) defined.


In [34]:
import torch
import numpy as np
import pandas as pd
from collections import Counter

In [35]:
def edit_distance(s1, s2):
    """Calculate the edit distance (Levenshtein distance) between two strings."""
    # Use dynamic programming
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    return dp[m][n]

In [36]:
def lcs_length(s1, s2):
    """
    Calculates the length of the longest common subsequence of s1 and s2
    using the formula from the paper: LCS(c, r) = 1/2 * (|c| + |r| - ED(c, r))
    """
    ed = edit_distance(s1, s2)
    return 0.5 * (len(s1) + len(s2) - ed)

In [37]:
def calculate_f1_score(candidate_str, reference_strs):
    """
    Calculates the F1 score for a candidate string against a list of reference strings.
    Uses the method from the paper (Section 3.2).
    """
    if not reference_strs:
        return 0.0, reference_strs[0] if reference_strs else ""

    # Find the best matching reference (minimum edit distance)
    best_ref = reference_strs[0]
    best_ref_lcs = lcs_length(candidate_str, reference_strs[0])
    for ref in reference_strs[1:]:
        lcs = lcs_length(candidate_str, ref)
        # The paper uses ED in the tie-breaker, but typically LCS is maximized
        # We use the original script's logic: ref for which ED is minimum,
        # which corresponds to max LCS here due to the formula.
        # Let's re-calculate ED for tie-breaking if needed, but the formula implies max LCS.
        # Original script: if (len(ref) - 2*lcs) < (len(best_ref) - 2*best_ref_lcs):
        # len(ref) - 2*lcs = len(ref) - (len(ref) + len(cand) - ED(ref, cand)) = ED(ref, cand) - len(cand)
        # So minimizing (len(ref) - 2*lcs) is equivalent to minimizing ED(ref, cand) - len(cand).
        # Minimizing ED(ref, cand) alone is simpler and aligns with the paper description.
        # However, the formula LCS = (|c|+|r|-ED)/2 means max LCS corresponds to min ED.
        # Let's stick to finding max LCS as the primary criterion, which the formula implies.
        if lcs > best_ref_lcs:
            best_ref = ref
            best_ref_lcs = lcs

    if best_ref_lcs == 0:
        return 0.0, best_ref

    try:
        precision = best_ref_lcs / len(candidate_str)
        recall = best_ref_lcs / len(best_ref)
        if (precision + recall) == 0:
            f1 = 0.0
        else:
            f1 = 2 * (precision * recall) / (precision + recall)
    except ZeroDivisionError:
        # This should ideally not happen if best_ref_lcs is calculated correctly
        f1 = 0.0
        best_ref = reference_strs[0] # Fallback

    return f1, best_ref

In [38]:
def evaluate_model(model, test_df, search_function, search_kwargs=None, max_len=50):
    """
    Evaluates the model using a specified search function (greedy or beam).

    Args:
        model (nn.Module): The trained model.
        test_df (pd.DataFrame): The test dataset.
        search_function (callable): The decoding function (e.g., greedy_search, beam_search).
        search_kwargs (dict): Arguments for the search function (e.g., {'beam_width': 5}).
        max_len (int): Maximum length for generated sequences.

    Returns:
        tuple: A tuple containing:
            - list: List of predicted strings.
            - list: List of reference strings (first one from the test set).
            - float: Word-level exact accuracy.
            - float: Character-level F1 score (mean).
    """
    if search_kwargs is None:
        search_kwargs = {}

    predictions = []
    references = []
    correct_predictions = 0
    total_f1_score = 0.0

    print(f"Evaluating with {search_function.__name__}...")
    for idx, row in test_df.iterrows():
        src_word = row['native word'] # Replace with your actual column name if different
        tgt_word = row['english word'] # Replace with your actual column name if different

        # Convert source string to indices tensor
        src_indices = torch.tensor([src_char_to_idx.get(ch, UNK_IDX) for ch in src_word], dtype=torch.long)

        # Generate prediction using the specified search function
        pred_indices = search_function(model, src_indices, max_len=max_len, **search_kwargs)

        # Convert predicted indices back to string
        pred_str = "".join([tgt_idx_to_char[idx] for idx in pred_indices if idx not in [SOS_IDX, EOS_IDX, PAD_IDX]])

        predictions.append(pred_str)
        references.append(tgt_word)

        # Calculate Word-Level Exact Accuracy (Top-1 ACC)
        if pred_str == tgt_word:
            correct_predictions += 1

        # Calculate Character-Level F1 Score (Top-1 F-score)
        f1, _ = calculate_f1_score(pred_str, [tgt_word])
        total_f1_score += f1

    num_samples = len(test_df)
    word_accuracy = correct_predictions / num_samples if num_samples > 0 else 0.0
    mean_f1_score = total_f1_score / num_samples if num_samples > 0 else 0.0

    return predictions, references, word_accuracy, mean_f1_score

In [40]:
# --- Perform Evaluation ---

# 1. Evaluate with Greedy Search
greedy_preds, greedy_refs, greedy_acc, greedy_f1 = evaluate_model(
    model, test_df, greedy_search, search_kwargs={}
)

# 2. Evaluate with Beam Search (e.g., beam width 5)
beam_width = 5
beam_preds, beam_refs, beam_acc, beam_f1 = evaluate_model(
    model, test_df, beam_search, search_kwargs={'beam_width': beam_width}
)

# --- Print Results in a Table Format ---
print("\n" + "="*60)
print("Evaluation Results on Test Set")
print("="*60)
results_df = pd.DataFrame({
    "Metric": ["Word-Level Exact Accuracy (ACC)", "Character-Level F1 Score (Mean)"],
    "Greedy Search": [greedy_acc, greedy_f1],
    f"Beam Search (beam={beam_width})": [beam_acc, beam_f1]
})

print(results_df.to_string(index=False, float_format='%.4f'))
print("="*60)

# Optional: Save predictions for manual inspection or further analysis
with open(f'greedy_predictions.txt', 'w', encoding='utf-8') as f:
    for pred, ref in zip(greedy_preds, greedy_refs):
        f.write(f"{pred}\t{ref}\n")
with open(f'beam_predictions_{beam_width}.txt', 'w', encoding='utf-8') as f:
    for pred, ref in zip(beam_preds, beam_refs):
        f.write(f"{pred}\t{ref}\n")

print("\nEvaluation completed.")

Evaluating with greedy_search...
Evaluating with beam_search...

Evaluation Results on Test Set
                         Metric  Greedy Search  Beam Search (beam=5)
Word-Level Exact Accuracy (ACC)         0.0130                0.0110
Character-Level F1 Score (Mean)         0.7470                0.7434

Evaluation completed.
